# LeetCode 3838 — Weighted Word Mapping bằng Python

Link đề: https://leetcode.com/problems/weighted-word-mapping/ · Độ khó: **Easy** · Chủ đề: Array, String, Simulation

Skeleton LeetCode:

```python
class Solution:
    def mapWordWeights(self, words: List[str], weights: List[int]) -> str:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua leetcode.doocs.org (Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình nên cần đối chiếu từng chữ với đề gốc nếu muốn chắc chắn.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi chữ cái `a..z` có một **trọng số**; trọng số của từ là tổng trọng số các chữ; lấy phần dư khi chia 26 rồi ánh xạ **ngược bảng chữ cái** (`0 -> 'z'`, `1 -> 'y'`, ..., `25 -> 'a'`).
2. Biết đổi chữ cái thành chỉ số bằng `ord(c) - ord('a')` và đổi chỉ số ngược lại bằng `chr`.
3. Biết dùng `dict`, `zip`, chuỗi tra cứu (`lookup string`) và `%` (modulo) để chuyển giá trị về miền `0..25`.
4. Viết 5 cách (vòng lặp tường minh, `dict` trọng số, bảng đảo chuỗi, một dòng `join` + generator, mod từng bước) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: ánh xạ xuôi thay vì ngược, quên `% 26`, quên trừ `ord('a')`, nhầm mod âm.

## 2. Đề bài & ví dụ

Cho mảng `words` gồm các từ chữ thường và mảng `weights` có đúng 26 số, `weights[i]` là trọng số của chữ cái thứ `i` (`weights[0]` cho `'a'`, ..., `weights[25]` cho `'z'`).

Với mỗi từ:

1. Tính **trọng số của từ** = tổng trọng số của mọi chữ trong từ.
2. Lấy `r = tổng % 26`.
3. Ánh xạ `r` thành một chữ cái theo **thứ tự bảng chữ cái đảo ngược**: `0 -> 'z'`, `1 -> 'y'`, ..., `25 -> 'a'`.

Nối các chữ cái thu được của mọi từ (đúng thứ tự trong `words`) thành một chuỗi và trả về.

| Ví dụ | words | weights | Output | Giải thích |
|---|---|---|---|---|
| 1 | `["abcd","def","xyz"]` | `[5,3,12,14,1,2,3,2,10,6,6,9,7,8,7,10,8,9,6,9,9,8,3,7,7,2]` | `"rij"` | `abcd` = 34 -> `34 % 26 = 8` -> `'r'`; `def` = 17 -> `'i'`; `xyz` = 16 -> `'j'` |
| 2 | `["a","b","c"]` | 26 số `1` | `"yyy"` | Mỗi từ nặng 1 -> `1 -> 'y'` |
| 3 | `["abcd"]` | `[7,5,3,4,3,5,4,9,4,2,2,7,10,2,5,10,6,1,2,2,4,1,3,4,4,5]` | `"g"` | `abcd` = 19 -> `19 -> 'g'` |

Constraints: `1 <= words.length <= 100`; `1 <= words[i].length <= 10`; `weights.length == 26`; `1 <= weights[i] <= 100`; `words[i]` chỉ gồm chữ cái thường.

In [1]:
W1 = [5,3,12,14,1,2,3,2,10,6,6,9,7,8,7,10,8,9,6,9,9,8,3,7,7,2]
W3 = [7,5,3,4,3,5,4,9,4,2,2,7,10,2,5,10,6,1,2,2,4,1,3,4,4,5]
for word, w in [("abcd", W1), ("def", W1), ("xyz", W1), ("abcd", W3)]:
    total = sum(w[ord(c) - ord('a')] for c in word)
    print(word, "tổng =", total, "mod 26 =", total % 26)

abcd tổng = 34 mod 26 = 8
def tổng = 17 mod 26 = 17
xyz tổng = 16 mod 26 = 16
abcd tổng = 19 mod 26 = 19


## 3. Kiến thức nền

### 3.1. `ord` và `chr`: chữ cái <-> số

`ord('a')` là `97`, `ord('b')` là `98`, ... `ord('z')` là `122`. `chr(97)` là `'a'`. Do đó `ord(c) - ord('a')` cho vị trí `0..25` của chữ `c` trong bảng chữ cái.

In [2]:
print(ord('a'), ord('b'), ord('z'))
print(chr(97), chr(122))
print([ord(c) - ord('a') for c in "abz"])

97 98 122
a z
[0, 1, 25]


### 3.2. Phép chia lấy dư `%`

`a % 26` đưa mọi số nguyên không âm về khoảng `0..25`. Ví dụ `34 % 26 = 8`, `26 % 26 = 0`, `19 % 26 = 19`. Ở bài này tổng không âm nên `%` không gặp rắc rối với số âm.

In [3]:
print(34 % 26, 26 % 26, 19 % 26, 1000 % 26, 52 % 26)

8 0 19 12 0


### 3.3. Ánh xạ ngược bảng chữ cái

Ta cần `0 -> 'z'`, `1 -> 'y'`, ..., `25 -> 'a'`. Có hai cách viết:

- Phép tính: `chr(ord('z') - r)`.
- Tra cứu: `'zyxwvutsrqponmlkjihgfedcba'[r]` (chuỗi đảo ngược của bảng chữ cái, truy cập theo chỉ số).

In [4]:
import string
rev = string.ascii_lowercase[::-1]
print(rev)
for r in [0, 1, 8, 19, 25]:
    print(r, chr(ord('z') - r), rev[r])

zyxwvutsrqponmlkjihgfedcba
0 z z
1 y y
8 r r
19 g g
25 a a


### 3.4. `dict` và `zip`

`zip(a, b)` ghép từng cặp phần tử; `dict(zip(keys, values))` tạo từ điển. Ta có thể tạo ngay `{'a': w0, 'b': w1, ...}` từ `string.ascii_lowercase` và `weights`.

In [5]:
import string
weights = [5,3,12,14,1,2,3,2,10,6,6,9,7,8,7,10,8,9,6,9,9,8,3,7,7,2]
wmap = dict(zip(string.ascii_lowercase, weights))
print(wmap['a'], wmap['d'], wmap['z'])
print(sum(wmap[c] for c in "abcd"))

5 14 2
34


### 3.5. `sum` với generator và `''.join`

`sum(f(c) for c in word)` cộng các giá trị mà không tạo list trung gian. `''.join(list_of_chars)` nối nhiều chuỗi một ký tự thành một chuỗi.

In [6]:
print(sum(ord(c) - 96 for c in "abc"))     # 1 + 2 + 3
print(''.join(["r", "i", "j"]))
print(''.join(c.upper() for c in "abc"))

6
rij
ABC


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Bài này là **mô phỏng** (simulation): làm đúng từng bước đề nói, không có mẹo. Với mỗi từ ta cần ba phép:

1. đổi từng chữ thành trọng số (tra mảng bằng chỉ số `ord(c) - 97`),
2. cộng lại và lấy dư mod 26,
3. đổi số `0..25` thành chữ theo bảng ngược.

Tính tay ví dụ 2: `words = ["a","b","c"]`, mọi trọng số là `1`. Từ `"a"` có tổng `1`, `1 % 26 = 1`, bảng ngược cho `1 -> 'y'`. Tương tự `"b"` và `"c"`. Kết quả `"yyy"`.

Tính tay ví dụ 1, từ `"def"`: `d` là chữ thứ 3 (`weights[3] = 14`), `e` là chữ thứ 4 (`weights[4] = 1`), `f` là chữ thứ 5 (`weights[5] = 2`). Tổng `17`, `17 % 26 = 17`, `chr(ord('z') - 17) = chr(105) = 'i'`.

| từ | chữ -> chỉ số | trọng số từng chữ | tổng | mod 26 | chữ |
|---|---|---|---|---|---|
| `abcd` | 0,1,2,3 | 5,3,12,14 | 34 | 8 | `r` |
| `def` | 3,4,5 | 14,1,2 | 17 | 17 | `i` |
| `xyz` | 23,24,25 | 7,7,2 | 16 | 16 | `j` |

In [7]:
W1 = [5,3,12,14,1,2,3,2,10,6,6,9,7,8,7,10,8,9,6,9,9,8,3,7,7,2]
for word in ["abcd", "def", "xyz"]:
    idx = [ord(c) - 97 for c in word]
    ws = [W1[i] for i in idx]
    total = sum(ws)
    r = total % 26
    print(f"{word}: chỉ số={idx} trọng số={ws} tổng={total} mod={r} chữ={chr(ord('z') - r)!r}")

abcd: chỉ số=[0, 1, 2, 3] trọng số=[5, 3, 12, 14] tổng=34 mod=8 chữ='r'
def: chỉ số=[3, 4, 5] trọng số=[14, 1, 2] tổng=17 mod=17 chữ='i'
xyz: chỉ số=[23, 24, 25] trọng số=[7, 7, 2] tổng=16 mod=16 chữ='j'


## 5. Cách 1 — Brute Force / baseline

Viết tường minh từng bước bằng hai vòng `for`: vòng ngoài qua từng từ, vòng trong qua từng chữ. Tích lũy `total`, lấy `% 26`, đổi thành chữ rồi `append` vào danh sách kết quả.

Đây là cách thẳng nhất và cũng đã tối ưu về bậc độ phức tạp (mỗi chữ cái chỉ phải đọc một lần). "Nút thắt" duy nhất là cách ta tra trọng số và tạo chữ: nếu tra bằng cách tìm kiếm trong bảng chữ cái (ví dụ `alphabet.index(c)`) thì mỗi lần tốn `O(26)`; ở đây dùng `ord` nên tra là `O(1)`.

In [8]:
class SolutionBrute:
    def mapWordWeights(self, words, weights):
        res = []
        for word in words:
            total = 0
            for c in word:
                total += weights[ord(c) - ord('a')]
            r = total % 26
            res.append(chr(ord('z') - r))
        return ''.join(res)

W1 = [5,3,12,14,1,2,3,2,10,6,6,9,7,8,7,10,8,9,6,9,9,8,3,7,7,2]
W3 = [7,5,3,4,3,5,4,9,4,2,2,7,10,2,5,10,6,1,2,2,4,1,3,4,4,5]
print(SolutionBrute().mapWordWeights(["abcd","def","xyz"], W1),
      SolutionBrute().mapWordWeights(["a","b","c"], [1]*26),
      SolutionBrute().mapWordWeights(["abcd"], W3))

rij yyy g


**Độ phức tạp:** Time `O(L)` với `L` là tổng số ký tự của mọi từ (`L <= 1000`), Space `O(m)` cho chuỗi kết quả (`m = len(words)`).

## 6. Ý tưởng thuật toán / bước đột phá

Vì đề đã là một quy trình tính toán rõ ràng, "đột phá" ở đây là **tổ chức việc tra cứu gọn gàng**:

- **Bảng trọng số theo chữ**: dựng `dict` một lần để mỗi lần dùng không phải nhớ công thức `ord(c) - 97`.
- **Bảng đảo ngược dạng chuỗi**: `rev = 'zyx...a'` nên chữ kết quả là `rev[r]`, không cần nhớ `ord('z') - r`.
- **Mod có thể áp dụng bất kỳ lúc nào**: `(a + b) % 26 = ((a % 26) + (b % 26)) % 26`. Nên lấy mod từng bước hoặc cuối cùng đều cho cùng kết quả. Trong Python phép `%` với số dương luôn cho kết quả `0..25`.

Invariant: sau mỗi từ, ta thêm **đúng một** chữ vào kết quả, nên `len(kết quả) == số từ đã xử lý`.

## 7. Các cách tiếp theo

### Cách 2 — `dict` trọng số theo chữ cái

Dựng `wmap = dict(zip(ascii_lowercase, weights))`, sau đó `sum(wmap[c] for c in word)`.

In [9]:
import string

class SolutionDict:
    def mapWordWeights(self, words, weights):
        wmap = dict(zip(string.ascii_lowercase, weights))
        res = []
        for word in words:
            total = sum(wmap[c] for c in word)
            res.append(chr(ord('z') - total % 26))
        return ''.join(res)

print(SolutionDict().mapWordWeights(["abcd","def","xyz"], W1), SolutionDict().mapWordWeights(["abcd"], W3))

rij g


**Độ phức tạp:** Time `O(L + 26)`, Space `O(26 + m)`.

### Cách 3 — Chuỗi tra cứu đảo ngược

Tạo `rev = ascii_lowercase[::-1]` và lấy `rev[total % 26]`. Rõ ràng hơn phép trừ `ord`: đọc code là thấy ngay "chỉ số 0 là z".

In [10]:
class SolutionLookup:
    def mapWordWeights(self, words, weights):
        rev = string.ascii_lowercase[::-1]        # 'zyx...a'
        res = []
        for word in words:
            total = 0
            for c in word:
                total += weights[ord(c) - 97]
            res.append(rev[total % 26])
        return ''.join(res)

print(SolutionLookup().mapWordWeights(["a","b","c"], [1]*26), SolutionLookup().mapWordWeights(["abcd"], W3))

yyy g


**Độ phức tạp:** Time `O(L)`, Space `O(m)`.

### Cách 4 — Một dòng: generator lồng trong `join`

Gói toàn bộ vào một biểu thức: với mỗi từ `w`, `rev[sum(weights[ord(c) - 97] for c in w) % 26]`, rồi `join`. Ngắn nhưng phải đọc từ trong ra ngoài.

In [11]:
class SolutionOneLine:
    def mapWordWeights(self, words, weights):
        rev = string.ascii_lowercase[::-1]
        return ''.join(rev[sum(weights[ord(c) - 97] for c in w) % 26] for w in words)

print(SolutionOneLine().mapWordWeights(["abcd","def","xyz"], W1), SolutionOneLine().mapWordWeights(["a","b","c"], [1]*26))

rij yyy


**Độ phức tạp:** Time `O(L)`, Space `O(m)`.

### Cách 5 — Lấy mod ngay từng bước

Giữ tổng luôn trong `0..25` bằng cách `total = (total + w) % 26` ở mỗi chữ. Kết quả y hệt (xem mục 6), và hữu ích khi trọng số có thể rất lớn (ở các ngôn ngữ có tràn số). Ở bài này chỉ để minh họa tính chất của mod.

In [12]:
class SolutionModStep:
    def mapWordWeights(self, words, weights):
        out = []
        for word in words:
            total = 0
            for c in word:
                total = (total + weights[ord(c) - 97]) % 26   # luôn nằm trong 0..25
            out.append(chr(122 - total))                       # 122 = ord('z')
        return ''.join(out)

print(SolutionModStep().mapWordWeights(["abcd","def","xyz"], W1), SolutionModStep().mapWordWeights(["abcd"], W3))

rij g


**Độ phức tạp:** Time `O(L)`, Space `O(m)`.

## 8. TRACE từng bước

Trace Cách 1 với ví dụ 1. Với mỗi từ ta in từng chữ, trọng số và tổng tích lũy.

In [13]:
def trace_map(words, weights):
    res = []
    for word in words:
        total = 0
        print(f"-- từ {word!r}")
        for c in word:
            w = weights[ord(c) - 97]
            total += w
            print(f"   chữ {c!r}: chỉ số {ord(c) - 97}, trọng số {w}, tổng tạm = {total}")
        r = total % 26
        ch = chr(ord('z') - r)
        print(f"   {total} % 26 = {r} -> {ch!r}")
        res.append(ch)
    return ''.join(res)

assert trace_map(["abcd", "def", "xyz"], W1) == "rij"
assert trace_map(["abcd"], W3) == "g"

-- từ 'abcd'
   chữ 'a': chỉ số 0, trọng số 5, tổng tạm = 5
   chữ 'b': chỉ số 1, trọng số 3, tổng tạm = 8
   chữ 'c': chỉ số 2, trọng số 12, tổng tạm = 20
   chữ 'd': chỉ số 3, trọng số 14, tổng tạm = 34
   34 % 26 = 8 -> 'r'
-- từ 'def'
   chữ 'd': chỉ số 3, trọng số 14, tổng tạm = 14
   chữ 'e': chỉ số 4, trọng số 1, tổng tạm = 15
   chữ 'f': chỉ số 5, trọng số 2, tổng tạm = 17
   17 % 26 = 17 -> 'i'
-- từ 'xyz'
   chữ 'x': chỉ số 23, trọng số 7, tổng tạm = 7
   chữ 'y': chỉ số 24, trọng số 7, tổng tạm = 14
   chữ 'z': chỉ số 25, trọng số 2, tổng tạm = 16
   16 % 26 = 16 -> 'j'
-- từ 'abcd'
   chữ 'a': chỉ số 0, trọng số 7, tổng tạm = 7
   chữ 'b': chỉ số 1, trọng số 5, tổng tạm = 12
   chữ 'c': chỉ số 2, trọng số 3, tổng tạm = 15
   chữ 'd': chỉ số 3, trọng số 4, tổng tạm = 19
   19 % 26 = 19 -> 'g'


In [14]:
# bảng trạng thái tổng hợp
rev = string.ascii_lowercase[::-1]
rows = []
for word in ["abcd", "def", "xyz"]:
    total = sum(W1[ord(c) - 97] for c in word)
    rows.append((word, total, total % 26, rev[total % 26]))
for row in rows:
    print("%-5s tổng=%2d mod=%2d chữ=%s" % row)
assert ''.join(r[3] for r in rows) == "rij"

abcd  tổng=34 mod= 8 chữ=r
def   tổng=17 mod=17 chữ=i
xyz   tổng=16 mod=16 chữ=j


## 9. Code nộp LeetCode

Dùng chuỗi tra cứu đảo ngược (Cách 3) với vòng lặp rõ ràng.

In [15]:
from typing import List, Optional


class Solution:
    def mapWordWeights(self, words: List[str], weights: List[int]) -> str:
        rev = "zyxwvutsrqponmlkjihgfedcba"          # rev[0] = 'z', ..., rev[25] = 'a'
        res = []
        for word in words:
            total = 0
            for c in word:
                total += weights[ord(c) - ord('a')]
            res.append(rev[total % 26])
        return ''.join(res)


print(Solution().mapWordWeights(["abcd", "def", "xyz"], [5,3,12,14,1,2,3,2,10,6,6,9,7,8,7,10,8,9,6,9,9,8,3,7,7,2]))

rij


## 10. Vì sao thuật toán đúng

1. **Tra trọng số đúng.** Với chữ thường `c`, `ord(c) - ord('a')` thuộc `0..25` và khớp quy ước `weights[0]` cho `'a'` ... `weights[25]` cho `'z'`.
2. **Trọng số từ.** Tổng là phép cộng thông thường trên các số nguyên dương, đúng định nghĩa.
3. **Ánh xạ ngược.** `rev[r]` với `r` trong `0..25`: `rev[0] = 'z'`, `rev[25] = 'a'`, đúng bảng của đề. `total % 26` luôn trong `0..25` (vì `total > 0`).
4. **Thứ tự.** Duyệt `words` từ trái sang phải và `append` mỗi từ một chữ nên chuỗi cuối đúng thứ tự đề yêu cầu.
5. **Mod từng bước = mod cuối.** `(a + b) mod m = ((a mod m) + (b mod m)) mod m`, nên Cách 5 cho cùng kết quả với các cách còn lại.

## 11. Phân tích Time / Space Complexity

Gọi `m = len(words)`, `L` là tổng độ dài các từ (`L <= 10 * 100 = 1000`).

- Mọi cách: **Time `O(L)`** vì mỗi chữ được đọc đúng một lần; phần dựng `dict` (Cách 2) thêm `O(26)` cố định.
- **Space `O(m)`** cho danh sách chữ kết quả (`O(1)` phụ nếu chỉ tính thêm biến đếm).

Không có sự khác biệt best / average / worst vì thuật toán không phụ thuộc cấu trúc dữ liệu, chỉ phụ thuộc tổng độ dài.

## 12. Bộ test

Đối chiếu 5 cách với ví dụ đề, biên (từ một chữ, tổng đúng bằng bội số của 26, trọng số lớn nhất) và 300 input ngẫu nhiên hợp lệ.

In [16]:
import random

solutions = [SolutionBrute(), SolutionDict(), SolutionLookup(), SolutionOneLine(), SolutionModStep(), Solution()]

def reference(words, weights):
    out = []
    for w in words:
        s = sum(weights[ord(c) - 97] for c in w)
        out.append("abcdefghijklmnopqrstuvwxyz"[25 - s % 26])     # cách viết khác để làm oracle
    return ''.join(out)

fixed = [
    (["abcd", "def", "xyz"], W1, "rij"),
    (["a", "b", "c"], [1] * 26, "yyy"),
    (["abcd"], W3, "g"),
    (["z"], [26] * 26, "z"),                 # 26 % 26 = 0 -> 'z'
    (["a"], [25] + [1] * 25, "a"),           # 25 % 26 = 25 -> 'a'
    (["zzzzzzzzzz"], [100] * 26, chr(122 - 1000 % 26)),   # tổng 1000, 1000 % 26 = 12
]
for words, weights, expected in fixed:
    for s in solutions:
        assert s.mapWordWeights(list(words), list(weights)) == expected, (type(s).__name__, words)

random.seed(38380)
for _ in range(300):
    m = random.randint(1, 100)
    words = [''.join(random.choice(string.ascii_lowercase) for _ in range(random.randint(1, 10))) for _ in range(m)]
    weights = [random.randint(1, 100) for _ in range(26)]
    exp = reference(words, weights)
    assert len(exp) == m
    for s in solutions:
        assert s.mapWordWeights(words, weights) == exp
print("tất cả test OK")

tất cả test OK


### Test thuộc tính

(1) Độ dài kết quả luôn bằng `len(words)`. (2) Nếu cộng thêm 26 vào **mọi** trọng số thì kết quả của các từ có độ dài `L` không đổi khi `L * 26 % 26 == 0` — tức luôn không đổi. (3) Thay thế `words[i]` bằng cùng một từ lặp lại thì chữ kết quả giống nhau.

In [17]:
random.seed(3838)
for _ in range(200):
    words = [''.join(random.choice(string.ascii_lowercase) for _ in range(random.randint(1, 10))) for _ in range(random.randint(1, 30))]
    weights = [random.randint(1, 74) for _ in range(26)]
    base = Solution().mapWordWeights(words, weights)
    assert len(base) == len(words)
    shifted = [w + 26 for w in weights]               # vẫn <= 100
    assert Solution().mapWordWeights(words, shifted) == base
    twice = Solution().mapWordWeights(words + words, weights)
    assert twice == base + base
print("property test OK")

property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: ánh xạ xuôi thay vì ngược

Dùng `chr(ord('a') + r)` cho ra `0 -> 'a'`, trái với đề (`0 -> 'z'`).

In [18]:
def wrong_forward(words, weights):
    res = []
    for w in words:
        r = sum(weights[ord(c) - 97] for c in w) % 26
        res.append(chr(ord('a') + r))             # SAI: phải là chr(ord('z') - r)
    return ''.join(res)

print(wrong_forward(["a", "b", "c"], [1] * 26), "!=", Solution().mapWordWeights(["a", "b", "c"], [1] * 26))
assert wrong_forward(["a", "b", "c"], [1] * 26) == "bbb" and Solution().mapWordWeights(["a", "b", "c"], [1] * 26) == "yyy"

bbb != yyy


### Lỗi 2: quên `% 26`

Tổng lớn hơn 25 sẽ đẩy `ord('z') - total` ra ngoài bảng chữ cái (ví dụ trọng số `34` cho `chr(88) = 'X'`).

In [19]:
def wrong_no_mod(words, weights):
    res = []
    for w in words:
        total = sum(weights[ord(c) - 97] for c in w)
        res.append(chr(ord('z') - total))          # SAI: thiếu % 26
    return ''.join(res)

print(repr(wrong_no_mod(["abcd"], W1)), "!=", Solution().mapWordWeights(["abcd"], W1))
assert wrong_no_mod(["abcd"], W1) == "X" and Solution().mapWordWeights(["abcd"], W1) == "r"

'X' != r


### Lỗi 3: quên trừ `ord('a')` khi tra trọng số

`weights[ord(c)]` truy cập chỉ số 97..122, ngoài mảng 26 phần tử -> `IndexError`.

In [20]:
def wrong_no_offset(words, weights):
    return ''.join(chr(122 - sum(weights[ord(c)] for c in w) % 26) for w in words)   # SAI

try:
    wrong_no_offset(["abcd"], W1)
except IndexError as e:
    print("IndexError:", e)
assert Solution().mapWordWeights(["abcd"], W1) == "r"

IndexError: list index out of range


### Lỗi 4: dùng chỉ số của từ làm trọng số hoặc tính mod trên từng chữ rồi nối

Một số bạn hiểu nhầm đề thành "mỗi chữ cái ra một chữ" (tức mỗi chữ được ánh xạ riêng). Đề yêu cầu **mỗi từ** ra **một** chữ.

In [21]:
def wrong_per_letter(words, weights):
    res = []
    for w in words:
        for c in w:                                    # SAI: ánh xạ từng chữ riêng
            res.append(chr(122 - weights[ord(c) - 97] % 26))
    return ''.join(res)

out = wrong_per_letter(["abcd", "def", "xyz"], W1)
print(len(out), "!=", len(Solution().mapWordWeights(["abcd", "def", "xyz"], W1)))
assert len(out) == 10 and len(Solution().mapWordWeights(["abcd", "def", "xyz"], W1)) == 3

10 != 3


### Lỗi 5: tính tổng trọng số của **tất cả** các từ rồi mới mod

Nếu cộng mọi từ vào một `total` duy nhất, ta chỉ được một chữ duy nhất thay vì `m` chữ.

In [22]:
def wrong_global_total(words, weights):
    total = 0
    for w in words:
        for c in w:
            total += weights[ord(c) - 97]
    return chr(122 - total % 26)                       # SAI: chỉ ra một chữ

print(wrong_global_total(["a", "b", "c"], [1] * 26), "!=", Solution().mapWordWeights(["a", "b", "c"], [1] * 26))
assert wrong_global_total(["a", "b", "c"], [1] * 26) == "w" and Solution().mapWordWeights(["a", "b", "c"], [1] * 26) == "yyy"

w != yyy


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề dạng "làm theo các bước" -> **simulation**: chuyển từng câu của đề thành một dòng code, không cố tìm thuật toán cao siêu.
- Chữ cái <-> số: `ord(c) - ord('a')` và `chr(ord('a') + i)`; ánh xạ ngược: `chr(ord('z') - i)`.
- Muốn khỏi nhầm chiều ánh xạ, hãy viết `rev = string.ascii_lowercase[::-1]` rồi `rev[r]` và tự kiểm tra `rev[0] == 'z'`.
- Luôn thử đúng ví dụ 2 (`"yyy"`) bằng tay: nó kiểm tra cả mod và chiều ánh xạ.
- Quen thuộc với dạng "băm" (hash) đơn giản: tổng ký tự mod một số.

## 15. Biến thể của bài

1. **Reverse the mapping**: thay đổi bảng ánh xạ (ví dụ `0 -> 'a'`) hoặc dùng mod khác.
2. **Sum of Digits of String After Convert** (1945): chuyển chữ thành số rồi cộng.
3. **Find the Difference** (389), **Determine if String Halves Are Alike**: thao tác trên chữ cái bằng mã ASCII.
4. **Shifting Letters** (848): dịch chữ cái theo tổng hậu tố, cũng dùng mod 26.
5. **Caesar cipher / Rotate String**: mã hóa dịch vòng tròn trên bảng chữ cái.
6. Biến thể: trọng số cũng thay đổi theo vị trí chữ trong từ (nhân với `i + 1`).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | hai vòng `for` tường minh | `O(L)` | `O(m)` | Dễ đọc nhất, nộp bài |
| 2. `dict` | `zip(ascii_lowercase, weights)` | `O(L + 26)` | `O(26 + m)` | Khi thích tra cứu theo chữ |
| 3. Chuỗi đảo | `rev[total % 26]` | `O(L)` | `O(m)` | Tránh phép trừ `ord` dễ nhầm |
| 4. Một dòng | generator + `join` | `O(L)` | `O(m)` | Viết nhanh khi đã quen |
| 5. Mod từng bước | `(total + w) % 26` | `O(L)` | `O(m)` | Khi trọng số rất lớn hoặc cần giữ số nhỏ |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().mapWordWeights(["ab", "ba"], [2, 3] + [1] * 24)` ra gì?

**Bài 2 (tính tay):** với trọng số toàn `1`, từ `"zzzzz"` (5 chữ) cho chữ nào? Từ có 26 chữ giống nhau thì sao?

**Bài 3 (điền TODO):** hoàn thành `word_weight(word, weights)` trả về tổng trọng số của một từ.

**Bài 4 (tự viết lại):** viết `unmap(ch)` trả về **giá trị `r`** (0..25) tương ứng với chữ kết quả `ch` (nghịch đảo của ánh xạ `r -> chữ`).

**Bài 5 (mini challenge):** viết `map_word_weights_forward(words, weights)` với bảng ánh xạ **xuôi** (`0 -> 'a'`), rồi chứng minh bằng assert rằng `forward(...)[i]` và kết quả đề gốc luôn "đối xứng": chữ xuôi `a`-based và chữ ngược `z`-based cộng với nhau có `(ord(x) - 97) + (ord(y) - 97) == 25`.

In [23]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().mapWordWeights(["ab", "ba"], [2, 3] + [1] * 24))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def word_weight_todo(word, weights):
    total = 0
    # TODO: cộng weights[ord(c) - 97] cho từng chữ c
    return total

uu


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: `"ab"` có tổng `2 + 3 = 5`; `"ba"` cũng `5`. Bài 2: `"zzzzz"` có tổng `5`.

**Mức 2 (vừa):** Bài 3: giống vòng trong của Cách 1. Bài 4: nếu `ch = chr(ord('z') - r)` thì `r = ord('z') - ord(ch)`. Bài 5: chỉ khác bảng ánh xạ cuối cùng.

**Mức 3 (gần đáp án):** Bài 1: `5 -> chr(122 - 5) = 'u'`, kết quả `"uu"`. Bài 2: tổng 5 -> `'u'`; 26 chữ giống nhau tổng `26` -> `26 % 26 = 0` -> `'z'`. Bài 4: `ord('z') - ord(ch)`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2
assert Solution().mapWordWeights(["ab", "ba"], [2, 3] + [1] * 24) == "uu"
assert Solution().mapWordWeights(["zzzzz"], [1] * 26) == "u"
assert Solution().mapWordWeights(["a" * 10], [1] * 26) == chr(122 - 10)

# Bài 3
def word_weight(word, weights):
    return sum(weights[ord(c) - 97] for c in word)
assert word_weight("abcd", W1) == 34 and word_weight("xyz", W1) == 16

# Bài 4
def unmap(ch):
    return ord('z') - ord(ch)
assert unmap('z') == 0 and unmap('a') == 25 and unmap('r') == 8
for r in range(26):
    assert unmap(chr(122 - r)) == r

# Bài 5
def map_word_weights_forward(words, weights):
    return ''.join(chr(97 + word_weight(w, weights) % 26) for w in words)
random.seed(19)
for _ in range(100):
    words = [''.join(random.choice(string.ascii_lowercase) for _ in range(random.randint(1, 10))) for _ in range(random.randint(1, 20))]
    weights = [random.randint(1, 100) for _ in range(26)]
    fw = map_word_weights_forward(words, weights)
    bw = Solution().mapWordWeights(words, weights)
    assert all((ord(x) - 97) + (ord(y) - 97) == 25 for x, y in zip(fw, bw))
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "mô phỏng tổng ký tự rồi ánh xạ bảng chữ cái":**

```python
rev = "zyxwvutsrqponmlkjihgfedcba"         # ánh xạ ngược: rev[0] = 'z'
res = []
for word in words:
    total = sum(weights[ord(c) - ord('a')] for c in word)
    res.append(rev[total % 26])
return ''.join(res)
```

**Checklist trước khi submit:**

- [ ] Chỉ số trọng số là `ord(c) - ord('a')` (0..25)?
- [ ] Có `% 26` và dùng bảng ánh xạ **ngược** (`0 -> 'z'`)?
- [ ] Mỗi từ cho **một** chữ, kết quả dài đúng `len(words)`?
- [ ] Đã thử ví dụ `"yyy"` và `"rij"` bằng tay?
- [ ] Chữ ký đúng `mapWordWeights(self, words: List[str], weights: List[int]) -> str`?